# CRUD Operations in ChromaDB (Adding and Querying Documents)**Level:** Scratch → Intermediate**Runs on:** Google Colab (Free CPU)**CRUD** = **C**reate, **R**ead, **U**pdate, **D**elete — the four basic operations every database supports.### What you will learn1. **Create** — set up a client and collection2. **Create (Add)** — insert documents with ids and metadata3. **Read (Query)** — semantic search + get-by-id + filters4. **Update** — modify existing documents/metadata5. **Delete** — remove documents by id or by filter6. Checking collection state (`count`, `peek`, `list_collections`)

### 1. Install ChromaDBRun this once.

In [1]:
# Install ChromaDB with OpenTelemetry versions compatible with Google ADK in Colab
!pip install -q chromadb "opentelemetry-api<=1.42.1,>=1.39" "opentelemetry-sdk<=1.42.1,>=1.39"

print("ChromaDB and compatible dependencies installed successfully!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
ChromaDB and compatible dependencies installed successfully!


### 2. CREATE — Client and Collection

In [2]:
import chromadb

# In-memory client (resets when runtime restarts) - great for learning
client = chromadb.Client()

# Create or reuse the collection so the cell can be safely rerun
collection = client.get_or_create_collection(name="crud_demo")

print("Collection created:", collection.name)
print("Current document count:", collection.count())


Collection created: crud_demo
Current document count: 0


### 3. CREATE — Adding Documents (`add`)Each document needs a unique `id`. `metadata` and `documents` are optional extras that describe each entry.

In [3]:
collection.upsert(
    documents=[
        "Python is a popular programming language for AI.",
        "The weather today is sunny and warm.",
        "Machine learning models learn patterns from data.",
        "I enjoy playing football on weekends.",
        "Deep learning uses neural networks with many layers."
    ],
    metadatas=[
        {"category": "tech"},
        {"category": "weather"},
        {"category": "tech"},
        {"category": "sports"},
        {"category": "tech"}
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"]
)

print("Documents added! Total count:", collection.count())


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 37.4MiB/s]


Documents added! Total count: 5


**Discussion Point:** If you try to `add()` an `id` that already exists, ChromaDB will raise an error. Use `upsert()` (shown later) if you want "add or update" behavior instead.

### 4. READ — Peek at the Collection`peek()` shows a quick sample of what's inside, without needing a query.

In [4]:
preview = collection.peek(limit=3)

for doc_id, doc in zip(preview["ids"], preview["documents"]):
    print(doc_id, "->", doc)


doc1 -> Python is a popular programming language for AI.
doc2 -> The weather today is sunny and warm.
doc3 -> Machine learning models learn patterns from data.


### 5. READ — Get Documents by IDUse `get()` when you know exactly which id(s) you want — this is an exact lookup, not a similarity search.

In [5]:
result = collection.get(ids=["doc1", "doc3"])

for doc_id, doc, meta in zip(
    result["ids"],
    result["documents"],
    result["metadatas"]
):
    print(f"{doc_id} | {meta} | {doc}")


doc1 | {'category': 'tech'} | Python is a popular programming language for AI.
doc3 | {'category': 'tech'} | Machine learning models learn patterns from data.


### 6. READ — Semantic Search (`query`)This is the core "vector database" feature: search by **meaning**, not exact words.

In [6]:
results = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=2
)

for doc_id, doc, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["distances"][0]
):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")


doc5 | distance=0.9317 | Deep learning uses neural networks with many layers.
doc1 | distance=0.9516 | Python is a popular programming language for AI.


### 7. READ — Filtering with `where` (Metadata Filter)Combine semantic search with exact filters — e.g., "only search within the 'tech' category".

In [7]:
results = collection.query(
    query_texts=["something fun to do outside"],
    n_results=1,
    where={"category": "sports"}
)

for doc_id, doc, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["distances"][0]
):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")


doc4 | distance=1.3895 | I enjoy playing football on weekends.


### 8. UPDATE — Modifying Existing Documents`update()` changes the text and/or metadata of documents that **already exist** (matched by `id`).

In [8]:
collection.update(
    ids=["doc4"],
    documents=["I enjoy playing football and basketball on weekends."],
    metadatas=[{"category": "sports", "updated": True}]
)

updated = collection.get(ids=["doc4"])

print("Updated document:", updated["documents"])
print("Updated metadata:", updated["metadatas"])


Updated document: ['I enjoy playing football and basketball on weekends.']
Updated metadata: [{'category': 'sports', 'updated': True}]


### 9. UPSERT — Add or Update in One Step`upsert()` is useful when you're not sure if the `id` already exists: if it exists, it updates it; if not, it creates it.

In [9]:
collection.upsert(
    documents=[
        "Deep learning uses neural networks with many layers and GPUs.",
        "The Eiffel Tower is located in Paris."
    ],
    ids=["doc5", "doc6"]
)

print("Total count after upsert:", collection.count())
print("Updated/new documents:", collection.get(ids=["doc5", "doc6"])["documents"])


Total count after upsert: 6
Updated/new documents: ['Deep learning uses neural networks with many layers and GPUs.', 'The Eiffel Tower is located in Paris.']


### 10. DELETE — Removing Documents by ID

In [10]:
collection.delete(ids=["doc2"])

print("Total count after delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])


Total count after delete: 5
Remaining ids: ['doc1', 'doc3', 'doc4', 'doc5', 'doc6']


### 11. DELETE — Removing Documents by FilterYou can also delete every document that matches a metadata filter, without listing ids manually.

In [11]:
collection.delete(where={"category": "tech"})

print("Total count after category delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])


Total count after category delete: 2
Remaining ids: ['doc4', 'doc6']


### 12. Verifying Everything (Final State Check)

In [12]:
final_state = collection.get()

print("Final document count:", collection.count())

for doc_id, doc, meta in zip(
    final_state["ids"],
    final_state["documents"],
    final_state["metadatas"]
):
    print(f"{doc_id} | {meta} | {doc}")


Final document count: 2
doc4 | {'category': 'sports', 'updated': True} | I enjoy playing football and basketball on weekends.
doc6 | None | The Eiffel Tower is located in Paris.


### 13. Key Takeaways (Recap for Students)| Operation | Method | Notes ||---|---|---|| Create | `client.create_collection()` | Makes a new collection (errors if it already exists) || Create/Add | `collection.add()` | Insert new documents; errors on duplicate `id` || Read (exact) | `collection.get()` | Fetch by `id`, no similarity involved || Read (semantic) | `collection.query()` | Search by meaning, returns closest matches + distances || Read (preview) | `collection.peek()` | Quick look at a few stored documents || Update | `collection.update()` | Modify existing document/metadata by `id` || Add or Update | `collection.upsert()` | Creates if missing, updates if it exists || Delete | `collection.delete()` | Remove by `id` list or by `where` metadata filter |### 14. Practice Exercises1. Add 5 new documents, then use `upsert()` to update 2 of them and create 1 brand-new one in a single call.2. Delete all documents where `category` equals a value of your choice, then confirm with `collection.count()`.3. Write a small function `safe_add(collection, id, text)` that uses `upsert()` internally so it never raises a duplicate-id error.

### Exercise 1 — Add 5 documents, then upsert 2 updates and 1 new document


In [13]:
exercise1_collection = client.get_or_create_collection(
    name="crud_exercise1"
)

# Add 5 new documents
exercise1_collection.upsert(
    documents=[
        "Python is useful for data science.",
        "Java is used for software development.",
        "Football is a popular team sport.",
        "Rain is expected tomorrow.",
        "Machine learning learns patterns from data."
    ],
    metadatas=[
        {"category": "technology"},
        {"category": "technology"},
        {"category": "sports"},
        {"category": "weather"},
        {"category": "technology"}
    ],
    ids=["ex1", "ex2", "ex3", "ex4", "ex5"]
)

# Update 2 existing documents and create 1 new document
exercise1_collection.upsert(
    documents=[
        "Python is widely used for data science and machine learning.",
        "Java is widely used for enterprise software development.",
        "Deep learning uses neural networks to learn complex patterns."
    ],
    metadatas=[
        {"category": "technology", "updated": True},
        {"category": "technology", "updated": True},
        {"category": "technology", "new": True}
    ],
    ids=["ex1", "ex2", "ex6"]
)

print("Documents after upsert:", exercise1_collection.count())
print(
    exercise1_collection.get(
        ids=["ex1", "ex2", "ex6"]
    )["documents"]
)


Documents after upsert: 6
['Python is widely used for data science and machine learning.', 'Java is widely used for enterprise software development.', 'Deep learning uses neural networks to learn complex patterns.']


**Observation:** `upsert()` updates existing IDs (`ex1`, `ex2`) and creates the new ID (`ex6`) in the same operation.


### Exercise 2 — Delete documents by a category and confirm the count


In [14]:
exercise2_collection = client.get_or_create_collection(
    name="crud_exercise2"
)

exercise2_collection.upsert(
    documents=[
        "Python is used in machine learning.",
        "Java is used for application development.",
        "Football is played by teams.",
        "Basketball is played on a court.",
        "The weather is sunny today."
    ],
    metadatas=[
        {"category": "technology"},
        {"category": "technology"},
        {"category": "sports"},
        {"category": "sports"},
        {"category": "weather"}
    ],
    ids=["e2_1", "e2_2", "e2_3", "e2_4", "e2_5"]
)

print("Count before delete:", exercise2_collection.count())

# Delete all sports documents
exercise2_collection.delete(
    where={"category": "sports"}
)

print("Count after deleting sports:", exercise2_collection.count())
print("Remaining IDs:", exercise2_collection.get()["ids"])


Count before delete: 5
Count after deleting sports: 3
Remaining IDs: ['e2_1', 'e2_2', 'e2_5']


**Observation:** The documents with `category = "sports"` are removed, and the final count confirms the deletion.


### Exercise 3 — Create a `safe_add()` function using `upsert()`


In [15]:
def safe_add(collection, id, text):
    """Add or update a document without raising a duplicate-ID error."""
    collection.upsert(
        ids=[id],
        documents=[text]
    )
    print(f"Document '{id}' added or updated successfully.")


exercise3_collection = client.get_or_create_collection(
    name="crud_exercise3"
)

safe_add(
    exercise3_collection,
    "safe1",
    "Python is a programming language."
)

# Calling the function again with the same ID updates the document
safe_add(
    exercise3_collection,
    "safe1",
    "Python is widely used for AI and data science."
)

print(exercise3_collection.get(ids=["safe1"]))


Document 'safe1' added or updated successfully.
Document 'safe1' added or updated successfully.
{'ids': ['safe1'], 'embeddings': None, 'documents': ['Python is widely used for AI and data science.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [None]}


**Observation:** `safe_add()` uses `upsert()`, so the same ID can be used repeatedly without a duplicate-ID error.
